<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Regression adjustment

**[Regression adjustment](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/)** · Experiments and Causal Impact: Did Our Change Work? · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** estimate the recommendation block's effect from logs (no test) by adding the confounders of the causal graph to a regression, and to see what a missing confounder, a mediator and a collider do to the answer.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Confounders, mediators and colliders, from [Draw the causes](https://learning.nextia-ai.com/courses/experiments/m01/draw-the-causes/); linear regression, from the [mathematics course](https://learning.nextia-ai.com/courses/math/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M06-L01-regression-adjustment/regression-adjustment-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "logs/customers.csv.gz": "0e9bd54ac28ef98bf94f6d1d0abe1bf0f8e51f67384d7b08c94b0b9310ab320a",
    "logs/truth.json": "8e2cd0eda8b31c1fba87687dfaf14b598d24921cf38734e8d18a21d8126b009f"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The naive answer, as a regression

A regression of revenue on `used_block` alone gives the same number as "users minus non-users".

In [ ]:
logs = pd.read_csv("data/logs/customers.csv.gz")
logs["desktop"] = (logs.device == "desktop").astype(int)

def effect(formula):
    m = smf.ols(formula, data=logs).fit(cov_type="HC1")
    b, se = m.params["used_block"], m.bse["used_block"]
    return f"{b:+.2f} dollars (95% interval {b - 1.96 * se:+.2f} to {b + 1.96 * se:+.2f})"

print("naive:", effect("revenue ~ used_block"))

> **Check.** You should see `naive: +9.70 dollars (95% interval +7.56 to +11.84)`.

## 2. Add the confounders

The causal graph of Module 1: past spending, past orders, visits, tenure, new or returning and the device cause both the use of the block and the spending. Add them.

In [ ]:
confounders = "pre_spend + pre_orders + visits + tenure_days + new_customer + desktop"
print("adjusted:", effect(f"revenue ~ used_block + {confounders}"))
t = truth("logs")
print(f"truth: {t['ate']['effect']:+.2f} dollars if everyone used it (ATE), {t['att']['effect']:+.2f} among the users (ATT)")

> **Check.** You should see `adjusted: +3.38 dollars (95% interval +1.74 to +5.03)` and `truth: +2.37 dollars if everyone used it (ATE), +2.73 among the users (ATT)`.

Much closer than the naive +9.70, but still above the truth. One confounder is not in the logs: whether the customer has a garden. No regression can adjust for a column it does not have (Module 6, lesson 3).

## 3. A missing confounder

> **Predict.** What happens if we forget past spending and visits?

In [ ]:
print("without pre_spend and visits:", effect("revenue ~ used_block + pre_orders + tenure_days + new_customer + desktop"))

> **Check.** You should see `without pre_spend and visits: +3.97 dollars (95% interval +2.07 to +5.87)`.

## 4. A mediator added by mistake

`pages_viewed` counts pages in the four weeks: the block itself adds pages. It is on the path from the block to spending, a **mediator**. Adding it takes away part of the effect we want to measure, and it mixes in the customers' intent to buy.

In [ ]:
print("with the mediator:", effect(f"revenue ~ used_block + {confounders} + pages_viewed"))

> **Check.** You should see `with the mediator: +2.79 dollars (95% interval +0.81 to +4.76)`.

Here it even lands closer to the truth, by luck: two errors cancel (the hidden garden pushes the estimate up, the mediator pulls it down). You cannot count on that. More control variables are not always better: a variable caused by the treatment does not belong in the regression.

## 5. Your turn: a collider

`left_review` is 1 if the customer wrote a product review. Both the block and spending cause reviews: it is a **collider**.

> **Your turn.** Fit the regression with the confounders **and** `left_review`. Put the coefficient of `used_block`, rounded to 2 decimals, in `with_collider`.

In [ ]:
with_collider = ...  # the used_block coefficient with the confounders and left_review, rounded to 2 decimals

In [ ]:
expect('with_collider', with_collider, -3.16)

Adjusting for a collider moves the estimate far from the truth, to the wrong sign: among customers who wrote a review, using the block and spending a lot become related for a reason that is not the block's effect.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/) has the graphs and the knowledge checks that count toward your certificate. In the project, `xp adjust` runs regression, matching and weighting together, and `xp adjust --add pages_viewed` shows the mediator's damage.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Propensity scores, matching and weighting](https://learning.nextia-ai.com/courses/experiments/m06/propensity-scores-matching-and-weighting/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m06/regression-adjustment/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.